In [2]:
import json
import logging
import mimetypes
import os
from argparse import Namespace
from http import HTTPStatus
from pathlib import Path
from typing import Any, Callable, Dict, List
from urllib.parse import urlencode
from wsgiref.simple_server import make_server

from sqllineage import DEFAULT_DIALECT, DEFAULT_HOST, DEFAULT_PORT, STATIC_FOLDER
from sqllineage.config import SQLLineageConfig
from sqllineage.core.metadata.dummy import DummyMetaDataProvider
from sqllineage.exceptions import SQLLineageException
from sqllineage.utils.constant import LineageLevel
from sqllineage.utils.helpers import extract_sql_from_args

logger = logging.getLogger(__name__)


class SQLLineageApp:
    """ 
    SQLLineageApp: A simple flask-like wsgi application to serve static files and handle lineage requests.
    """
    def __init__(self) -> None:
        # save route path 
        self.routes: Dict[str, Callable[[Dict[str, Any]], Dict[str, Any]]] = {}
        self.root_path = Path(SQLLineageConfig.DIRECTORY)
        self.metadata_provider = DummyMetaDataProvider()

    def route(self, path: str):
        def wrapper(handler):
            self.routes[path] = handler
            return handler

        return wrapper

    def __call__(self, environ, start_response) -> List[bytes]:
        static_folder = Path(os.path.dirname(__file__)).joinpath(Path(STATIC_FOLDER))
        request_method = environ["REQUEST_METHOD"]
        path_info = environ["PATH_INFO"]
        try:
            if request_method == "GET":
                mimetype = "text/html; charset=utf-8"
                if path_info == "/":
                    static_fname = str(static_folder.joinpath(Path("index.html")))
                else:
                    if ".." in path_info:
                        # Do not allow going back to parent path of static folder
                        return self.handle_404(start_response)
                    static_file = static_folder.joinpath(Path(path_info.strip("/")))
                    if static_file.exists():
                        static_fname = str(static_file)
                        optional_mimetype = mimetypes.guess_type(path_info)[0]
                        mimetype = (
                            optional_mimetype
                            if optional_mimetype is not None
                            else mimetype
                        )
                    else:
                        return self.handle_404(start_response)
                with open(static_fname, "rb") as f:
                    text = f.read()
                return self.handle_200_text(start_response, mimetype, text)
            elif request_method == "POST":
                print("routes:", self.routes)
                if path_info in self.routes:
                    request_body_size = int(environ["CONTENT_LENGTH"])
                    request_body = environ["wsgi.input"].read(request_body_size)
                    payload = json.loads(request_body)
                    for param in ["d", "f"]:
                        if param in payload and not str(
                            Path(payload[param]).absolute()
                        ).startswith(str(Path(self.root_path).absolute())):
                            return self.handle_403(start_response)
                    data = self.routes[path_info](payload)
                    # print("data:", data)
                    return self.handle_200_json(start_response, data)
                else:
                    return self.handle_404(start_response)
            elif request_method == "OPTIONS":
                if path_info in self.routes:
                    start_response(
                        "200 OK",
                        [
                            ("Access-Control-Allow-Origin", "*"),
                            (
                                "Access-Control-Allow-Headers",
                                "Content-Type",
                            ),
                            ("Access-Control-Allow-Methods", "POST"),
                        ],
                    )
                    return []
                else:
                    return self.handle_404(start_response)
            else:
                return self.handle_405(start_response)
        except (SystemExit, IsADirectoryError, FileNotFoundError, PermissionError):
            return self.handle_404(start_response)
        except (SQLLineageException, RuntimeError) as e:
            return self.handle_400(start_response, str(e))

    @staticmethod
    def handle_200_text(start_response, mimetype, text) -> List[bytes]:
        status_code = HTTPStatus.OK
        start_response(
            f"{status_code.value} {status_code.phrase}", [("Content-type", mimetype)]
        )
        return [text]

    def handle_200_json(self, start_response, data) -> List[bytes]:
        return self.handle_json_response(start_response, HTTPStatus.OK, data)

    def handle_400(self, start_response, message) -> List[bytes]:
        return self.handle_client_error_response(
            start_response, HTTPStatus.BAD_REQUEST, message
        )

    def handle_403(self, start_response) -> List[bytes]:
        message = "File Not Allowed For Accessing"
        return self.handle_client_error_response(
            start_response, HTTPStatus.FORBIDDEN, message
        )

    def handle_404(self, start_response) -> List[bytes]:
        message = "File Not Found"
        return self.handle_client_error_response(
            start_response, HTTPStatus.NOT_FOUND, message
        )

    def handle_405(self, start_response) -> List[bytes]:
        message = "Method Not Allowed"
        return self.handle_client_error_response(
            start_response, HTTPStatus.METHOD_NOT_ALLOWED, message
        )

    def handle_client_error_response(
        self, start_response, status_code, message
    ) -> List[bytes]:
        data = {"message": message}
        return self.handle_json_response(start_response, status_code, data)

    @staticmethod
    def handle_json_response(start_response, status_code, data) -> List[bytes]:
        start_response(
            f"{status_code.value} {status_code.phrase}",
            [
                ("Content-type", "application/json"),
                ("Access-Control-Allow-Origin", "*"),
            ],
        )
        return [json.dumps(data).encode("utf-8")]


app = SQLLineageApp()


@app.route("/lineage")
def lineage(payload):
    # this is to avoid circular import
    from sqllineage.runner import LineageRunner

    req_args = Namespace(**payload)
    sql = extract_sql_from_args(req_args)
    dialect = getattr(req_args, "dialect", DEFAULT_DIALECT)
    lr = LineageRunner(
        sql, dialect=dialect, verbose=True, metadata_provider=app.metadata_provider
    )
    data = {
        "verbose": str(lr),
        "dag": lr.to_cytoscape(),
        "column": lr.to_cytoscape(LineageLevel.COLUMN),
    }
    return data


@app.route("/script")
def script(payload):
    req_args = Namespace(**payload)
    sql = extract_sql_from_args(req_args)
    return {"content": sql}


@app.route("/directory")
def directory(payload):
    if payload.get("f"):
        root = Path(payload["f"]).parent
    elif payload.get("d"):
        root = Path(payload["d"])
    else:
        root = Path(SQLLineageConfig.DIRECTORY)
    data = {
        "id": str(root),
        "name": root.name,
        "is_dir": True,
        "children": [
            {"id": str(p), "name": p.name, "is_dir": p.is_dir()}
            for p in sorted(root.iterdir(), key=lambda _: (not _.is_dir(), _.name))
        ],
    }
    return data


def draw_lineage_graph(**kwargs) -> None:
    host = kwargs.pop("host", DEFAULT_HOST) 
    port = kwargs.pop("port", DEFAULT_PORT)
    querystring = urlencode({k: v for k, v in kwargs.items() if v}) # 将字典转换为url参数
    path = f"/?{querystring}" if querystring else "/" # 生成url
    if f := kwargs.get("f"): # 获取文件路径
        app.root_path = Path(f).parent # 设置文件路径
    if metadata_provider := kwargs.get("metadata_provider"): # 获取元数据
        app.metadata_provider = metadata_provider # 设置元数据
    with make_server(host, port, app) as httpd: # 启动服务 
        print(f" * SQLLineage Running on http://{host}:{port}{path}") # 打印服务地址
        httpd.serve_forever()  # 服务一直运行


In [3]:
## read sql file to string
sql_path = '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/tpcds/app_rt_trip_issue_detail_hf.sql'
with open(sql_path, 'r') as f:
    sql = f.read()

In [4]:
from sqllineage.runner import LineageRunner
result = LineageRunner(sql,dialect='non-validating')
print(result)

/tmp/ipykernel_2924427/2817020059.py:2: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(sql,dialect='non-validating')


Statements(#): 2
Source Tables:
    vgds.dim_rt_issue_topic_view_hf
    vgds.dim_rt_trip_distance_accumulated_df
    vgds.dim_rt_version_date_range_df
    vgds.dwd_rt3_task_order_package_case_order_hf
    vgds.dwd_rt_issue_with_merged_topic_detail_hf
    vgds.dwd_rt_trip_info_hf
    vgds.dwd_ssevent_data_quality_issue_detail_hf
    vgds.ods_rt_issue_info_1_hf
Target Tables:
    <default>.app_rt_trip_issue_detail_hf



In [5]:
result.draw()


 * SQLLineage Running on http://localhost:5001/?e=create+table+if+not+exists+%60app_rt_trip_issue_detail_hf%60%0A%28%0A++++%60car_id%60+string+COMMENT+%27%E8%BD%A6%E8%BE%86%E7%BC%96%E5%8F%B7%27%0A++++%2C%60trip_comment%60+string+COMMENT+%27comment%27%0A++++%2C%60country%60+bigint+COMMENT+%271%3Acn+2%3Aus%27%0A++++%2C%60driver_name%60+string+COMMENT+%27%E9%A9%BE%E9%A9%B6%E5%91%98%E5%90%8D%E7%A7%B0%27%0A++++%2C%60bag_trip_end_timestamp%60+bigint+COMMENT+%27bag_trip_end_timestamp%27%0A++++%2C%60region%60+string+COMMENT+%27%E5%9C%B0%E5%8C%BA%27%0A++++%2C%60bag_trip_start_timestamp%60+bigint+COMMENT+%27bag_trip_start_timestamp%27%0A++++%2C%60trip_id%60+string+COMMENT+%27trip_id%27%0A++++%2C%60update_time%60+string+COMMENT+%27%E6%9B%B4%E6%96%B0%E6%97%B6%E9%97%B4%27%0A++++%2C%60user_name%60+string+COMMENT+%27%E5%AE%89%E5%85%A8%E5%91%98%E5%90%8D%E7%A7%B0%27%0A++++%2C%60test_version%60+string+COMMENT+%27%E8%87%AA%E5%8A%A8%E9%A9%BE%E9%A9%B6%E7%89%88%E6%9C%AC%27%0A++++%2C%60road_test_type%60+bigi

127.0.0.1 - - [17/Sep/2024 00:40:11] "GET /?f=/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/tpcds/app_rt_trip_issue_detail_hf.sql HTTP/1.1" 200 739
127.0.0.1 - - [17/Sep/2024 00:40:11] "GET /static/js/main.3b88c6e3.js HTTP/1.1" 200 3184857
127.0.0.1 - - [17/Sep/2024 00:40:12] "GET /static/css/main.84d1d546.css HTTP/1.1" 200 65835
127.0.0.1 - - [17/Sep/2024 00:40:12] "GET /static/js/333.140e3456.chunk.js HTTP/1.1" 200 18267
127.0.0.1 - - [17/Sep/2024 00:40:12] "POST /directory HTTP/1.1" 200 13567
127.0.0.1 - - [17/Sep/2024 00:40:12] "POST /script HTTP/1.1" 200 19369


routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}
routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}
routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}


127.0.0.1 - - [17/Sep/2024 00:40:12] "POST /lineage HTTP/1.1" 200 74956
127.0.0.1 - - [17/Sep/2024 00:40:12] "POST /script HTTP/1.1" 200 19369


routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}
routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}


...

127.0.0.1 - - [17/Sep/2024 01:34:14] "POST /lineage HTTP/1.1" 200 394
127.0.0.1 - - [17/Sep/2024 01:34:14] "POST /script HTTP/1.1" 200 4539
127.0.0.1 - - [17/Sep/2024 01:34:14] "GET /editor.worker.js HTTP/1.1" 200 121291


routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}
routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}


127.0.0.1 - - [17/Sep/2024 01:34:16] "POST /lineage HTTP/1.1" 200 18196
127.0.0.1 - - [17/Sep/2024 01:34:16] "POST /script HTTP/1.1" 200 2301
127.0.0.1 - - [17/Sep/2024 01:34:16] "GET /editor.worker.js HTTP/1.1" 200 121291


routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}
routes: {'/lineage': <function lineage at 0x778e299d9940>, '/script': <function script at 0x778e299d99e0>, '/directory': <function directory at 0x778e299d9a80>}


127.0.0.1 - - [17/Sep/2024 01:34:19] "POST /script HTTP/1.1" 200 8829
127.0.0.1 - - [17/Sep/2024 01:34:19] "POST /lineage HTTP/1.1" 200 42263
127.0.0.1 - - [17/Sep/2024 01:34:19] "GET /editor.worker.js HTTP/1.1" 200 121291
127.0.0.1 - - [17/Sep/2024 01:34:40] "GET /static/media/codicon.4168b9c11e5075e9cfe6.ttf HTTP/1.1" 200 62792
